# N00 · 环境 Warmup

这个 notebook 可以在 CPU 环境运行，用于读取 L00 的 `collect_env.json`，理解 `LOCAL_RANK`、`CUDA_VISIBLE_DEVICES` 与设备映射。

## 1. 读取 collect_env JSON

如果你还没有运行 L00，下面会生成一个最小示例，帮助你先理解字段结构。

In [ ]:
import json
from pathlib import Path

candidates = sorted(Path("../runs/l01_env_conda_cuda").glob("*/artifacts/collect_env.json"))
if candidates:
    path = candidates[-1]
    payload = json.loads(path.read_text(encoding="utf-8"))
else:
    path = None
    payload = {"python": "unknown", "cuda_available": False, "cuda_device_count": 0, "note": "sample payload"}
print("source:", path or "sample")
payload


## 2. LOCAL_RANK 与 CUDA_VISIBLE_DEVICES

`CUDA_VISIBLE_DEVICES=3,5` 时，进程看到的本地 device 0 实际映射到物理 GPU 3；本地 device 1 映射到物理 GPU 5。

In [ ]:
def visible_mapping(cuda_visible_devices: str, local_rank: int):
    if not cuda_visible_devices:
        return {"local_rank": local_rank, "physical_device": local_rank, "note": "CUDA_VISIBLE_DEVICES not set"}
    visible = [item.strip() for item in cuda_visible_devices.split(",") if item.strip()]
    if local_rank >= len(visible):
        return {"local_rank": local_rank, "physical_device": None, "error": "LOCAL_RANK out of visible range"}
    return {"local_rank": local_rank, "physical_device": visible[local_rank]}

for env in ["", "0", "3,5", "7,2,4"]:
    print(env or "<unset>", "->", [visible_mapping(env, r) for r in range(2)])


## 3. 自检问题

- 如果 `LOCAL_RANK=1` 但 `CUDA_VISIBLE_DEVICES=0`，会发生什么？
- 为什么 rank0 写出 artifact 不代表所有 rank 都走到了 barrier？
- 哪些信息必须写入 `collect_env.json` 才能让别人复现？

In [ ]:
required_fields = ["python", "cuda_available", "cuda_device_count"]
missing = [field for field in required_fields if field not in payload]
print("missing fields:", missing)
print("ready_for_l00_report:", not missing)
